In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
# Setup paths
DATA_DIR = Path("../datasets/ProcessedData")
MODEL_DIR = Path("../models_sklearn")
MODEL_DIR.mkdir(exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print(f"Model directory: {MODEL_DIR}")

Data directory: ../datasets/ProcessedData
Model directory: ../models_sklearn


In [3]:
# Load preprocessed data
print("Loading data...")
train_df = pd.read_parquet(DATA_DIR / "train.parquet")
test_df = pd.read_parquet(DATA_DIR / "test.parquet")

print(f"Train: {len(train_df):,} samples")
print(f"Test: {len(test_df):,} samples")
print(f"\nColumns: {train_df.columns.tolist()}")

Loading data...
Train: 190,829 samples
Test: 47,696 samples

Columns: ['jaar_ongeval', 'aantal_partijen', 'maximum_snelheid', 'lon', 'lat', 'verkeersongeval_afloop', 'aard_ongeval', 'bebouwde_kom', 'wegverlichting', 'weersgesteldheid', 'features']


In [4]:
# Prepare features - exclude label columns
exclude_cols = [
    'verkeersongeval_afloop', 'aard_ongeval', 'bebouwde_kom',
    'verkeersongeval_afloop_idx', 'aard_ongeval_idx', 'bebouwde_kom_idx',
    'severity_label', 'type_label', 'location_label'
]

feature_cols = [col for col in train_df.columns if col not in exclude_cols]
X_train = train_df[feature_cols].select_dtypes(include=[np.number]).fillna(0)
X_test = test_df[feature_cols].select_dtypes(include=[np.number]).fillna(0)

print(f"Features: {X_train.shape[1]}")
print(f"Feature names: {X_train.columns.tolist()[:10]}...")  # Show first 10

Features: 5
Feature names: ['jaar_ongeval', 'aantal_partijen', 'maximum_snelheid', 'lon', 'lat']...


## 1. Train Severity Predictor (Logistic Regression)

In [5]:
print("=== Training Severity Model ===")

# Encode labels
le_severity = LabelEncoder()
y_train_severity = le_severity.fit_transform(train_df['verkeersongeval_afloop'])
y_test_severity = le_severity.transform(test_df['verkeersongeval_afloop'])

print(f"Classes: {le_severity.classes_}")

# Train model
lr_severity = LogisticRegression(max_iter=100, random_state=42, n_jobs=-1)
lr_severity.fit(X_train, y_train_severity)

# Evaluate
y_pred = lr_severity.predict(X_test)
accuracy = accuracy_score(y_test_severity, y_pred)
print(f"\nAccuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test_severity, y_pred, target_names=le_severity.classes_))

=== Training Severity Model ===
Classes: ['Dodelijk' 'Letsel' 'Uitsluitend materiele schade']

Accuracy: 0.7288

Classification Report:
                              precision    recall  f1-score   support

                    Dodelijk       0.00      0.00      0.00       319
                      Letsel       0.29      0.00      0.00     12601
Uitsluitend materiele schade       0.73      1.00      0.84     34776

                    accuracy                           0.73     47696
                   macro avg       0.34      0.33      0.28     47696
                weighted avg       0.61      0.73      0.62     47696



/Users/kooroshkz/Desktop/CrashScope/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/kooroshkz/Desktop/CrashScope/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/kooroshkz/Desktop/CrashSc

In [6]:
# Save severity model
joblib.dump(lr_severity, MODEL_DIR / "severity_model.pkl")
joblib.dump(le_severity, MODEL_DIR / "severity_encoder.pkl")
print("✓ Saved severity model and encoder")

✓ Saved severity model and encoder


## 2. Train Accident Type Classifier (Logistic Regression)

In [7]:
print("=== Training Accident Type Model ===")

# Encode labels
le_type = LabelEncoder()
y_train_type = le_type.fit_transform(train_df['aard_ongeval'])
y_test_type = le_type.transform(test_df['aard_ongeval'])

print(f"Classes: {le_type.classes_}")

# Train model
lr_type = LogisticRegression(max_iter=100, random_state=42, n_jobs=-1)
lr_type.fit(X_train, y_train_type)

# Evaluate
y_pred = lr_type.predict(X_test)
accuracy = accuracy_score(y_test_type, y_pred)
print(f"\nAccuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test_type, y_pred, target_names=le_type.classes_))

=== Training Accident Type Model ===
Classes: ['Dier' 'Eenzijdig' 'Flank' 'Frontaal' 'Geparkeerd voertuig' 'Kop/staart'
 'Los voorwerp' 'Onbekend' 'Vast voorwerp' 'Voetganger']

Accuracy: 0.3595

Classification Report:
                     precision    recall  f1-score   support

               Dier       0.00      0.00      0.00       275
          Eenzijdig       0.00      0.00      0.00      2933
              Flank       0.36      1.00      0.53     17145
           Frontaal       0.00      0.00      0.00      2893
Geparkeerd voertuig       0.00      0.00      0.00      1294
         Kop/staart       0.00      0.00      0.00      9950
       Los voorwerp       0.00      0.00      0.00       846
           Onbekend       0.00      0.00      0.00      6017
      Vast voorwerp       0.00      0.00      0.00      5205
         Voetganger       0.00      0.00      0.00      1138

           accuracy                           0.36     47696
          macro avg       0.04      0.10      0

/Users/kooroshkz/Desktop/CrashScope/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/kooroshkz/Desktop/CrashScope/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/kooroshkz/Desktop/CrashSc

In [8]:
# Save accident type model
joblib.dump(lr_type, MODEL_DIR / "accident_type_model.pkl")
joblib.dump(le_type, MODEL_DIR / "accident_type_encoder.pkl")
print("✓ Saved accident type model and encoder")

✓ Saved accident type model and encoder


## 3. Train Location Risk Assessor (Random Forest)

In [9]:
print("=== Training Location Risk Model ===")

# Encode labels
le_location = LabelEncoder()
y_train_location = le_location.fit_transform(train_df['bebouwde_kom'])
y_test_location = le_location.transform(test_df['bebouwde_kom'])

print(f"Classes: {le_location.classes_}")

# Train model
rf_location = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
rf_location.fit(X_train, y_train_location)

# Evaluate
y_pred = rf_location.predict(X_test)
accuracy = accuracy_score(y_test_location, y_pred)
print(f"\nAccuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test_location, y_pred, target_names=le_location.classes_))

=== Training Location Risk Model ===
Classes: ['Binnen' 'Buiten']

Accuracy: 0.9326

Classification Report:
              precision    recall  f1-score   support

      Binnen       0.94      0.96      0.95     31327
      Buiten       0.92      0.88      0.90     16369

    accuracy                           0.93     47696
   macro avg       0.93      0.92      0.92     47696
weighted avg       0.93      0.93      0.93     47696



In [10]:
# Save location risk model
joblib.dump(rf_location, MODEL_DIR / "location_risk_model.pkl")
joblib.dump(le_location, MODEL_DIR / "location_risk_encoder.pkl")
print("✓ Saved location risk model and encoder")

✓ Saved location risk model and encoder


## 4. Save Metadata

In [11]:
# Save feature names for consistency
joblib.dump(X_train.columns.tolist(), MODEL_DIR / "feature_names.pkl")
print(f"✓ Saved {len(X_train.columns)} feature names")

print("\n" + "="*50)
print("✓ All models trained and saved successfully!")
print(f"✓ Models directory: {MODEL_DIR.absolute()}")
print("="*50)

✓ Saved 5 feature names

✓ All models trained and saved successfully!
✓ Models directory: /Users/kooroshkz/Desktop/CrashScope/notebooks/../models_sklearn
